### This script should present the not matchable sequences of the design probably due to problems in the MPRAOligo design workflow
- expectation all header with REF_ or ALT_ should be in either ref_id or alt_id column of the variant region map
- read the design file with metadata (variants and elements)
- try to match with the ref and alt ids and the regions (copied from variant_region_map notebook)
- final design: 46374 variants

In [4]:
# imports
import pandas as pd 
import yaml

# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf

# read config
# config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/config/config.yaml"
config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/global80K_config.yaml"
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path, 'r') as ymlfile:
    config = yaml.safe_load(ymlfile)
    

# column names
col_name = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class' # SNP
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'

In [7]:
# load design file
pre_metadata_df_path = '/home/kisa/coding/80K_MPRA/design_data/metadata_example_2504.tsv'
pre_metadata_df = pd.read_csv(pre_metadata_df_path, sep="\t")

# # load variant file
# vcf_path = config['files']['final_design']['vcf_file']

# load variant region map
variant_region_map_path = config['files']['final_design']['final_design_variant_region_map']
variant_region_map = pd.read_csv(variant_region_map_path, sep="\t")
variant_region_map

# get bed in dataframe
bed_path = config['files']['final_design']['region_bed']
bed_df = pd.read_csv(bed_path, sep="\t", header=None)
bed_df.columns = [col_chr, col_start, col_end, 'ID', 'score', col_strand]

In [8]:
# filter everything only look at tested sequences
tested_variant_region_map = variant_region_map[variant_region_map['Variant'].str.startswith('cardiac_neuro_cava_random')]
print('only tested variants: ', tested_variant_region_map.shape[0])

tested_bed_df = bed_df[bed_df['ID'].str.startswith('cardiac_neuro_cava_random')]
print('only tested region rows: ', tested_bed_df.shape[0])

tested_pre_metadata_df = pre_metadata_df[pre_metadata_df['tmp_label'] == 'cardiac_neuro_cava_random']
# remove na columns which will be added in the following code
tested_pre_metadata_df = tested_pre_metadata_df.drop(columns=[col_chr, col_start, col_end, col_strand])
print('only tested metadata rows: ', tested_pre_metadata_df.shape[0])

only tested variants:  46374
only tested region rows:  27556
only tested metadata rows:  73940


In [9]:
tested_variant_region_map.head()

,Variant,Region,REF_ID,ALT_ID
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...


In [10]:
tested_bed_df.head()

,chr,start,end,ID,score,strand
0,chr1,2179507,2179777,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
1,chr1,2181843,2182113,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
2,chr1,2182439,2182709,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
3,chr1,2182830,2183100,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
4,chr1,2185027,2185297,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+


In [11]:
tested_pre_metadata_df.head()

,header,sequence,tmp_label,name,category,class,source,ref,variant_class,variant_pos,SPDI,allele,info,tmp_matching_header
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....


### Find edge cases: 94 have ref and alt but cannot be found in the ref and alt id column of the variant region map
- get sequences which are elements according to out metadata but have "REF_" or "ALT_" in the header
- None of these sequences can be found in the ref_id or alt_id of the final variant region map

In [25]:
elements_in_design = tested_pre_metadata_df.loc[tested_pre_metadata_df[col_category] == 'element']
not_fitting_cases = elements_in_design.loc[elements_in_design[col_name].str.contains('REF_|ALT_')]
print('Number of not fitting cases: ', not_fitting_cases.shape[0])
not_fitting_cases.head()
not_fitting_cases.to_csv('not_fitting_cases.tsv', sep="\t", index=False)

Number of not fitting cases:  94


In [16]:
# split in ref and alt
ref_not_fitting_cases = not_fitting_cases.loc[not_fitting_cases[col_name].str.contains(":REF_")]
alt_not_fitting_cases = not_fitting_cases.loc[not_fitting_cases[col_name].str.contains(":ALT_")]

In [19]:
tested_variant_region_map.head()

,Variant,Region,REF_ID,ALT_ID
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...


In [20]:
# investigate final design region map
ref_not_fitting_region = ref_not_fitting_cases.merge(tested_variant_region_map[['Region', 'REF_ID']].drop_duplicates(), left_on="tmp_matching_header", right_on="REF_ID", how="left")
merged_ref_not_fitting_region = ref_not_fitting_region[~ref_not_fitting_region['Region'].isna()]
print("matched ref: ", merged_ref_not_fitting_region.shape[0]) # 0

alt_not_fitting_region = alt_not_fitting_cases.merge(tested_variant_region_map[['Region', 'ALT_ID']].drop_duplicates(), left_on="tmp_matching_header", right_on="ALT_ID", how="left")
merged_alt_not_fitting_region = alt_not_fitting_region[~alt_not_fitting_region['Region'].isna()]
print("matched alt: ", merged_alt_not_fitting_region.shape[0]) # 0

matched ref:  0
matched alt:  0


In [24]:
# investigate unfiltered region map
unfiltered_variant_region_path = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/oligo_design/cardiac_neuro_cava_random/design_variants.region_map.tsv.gz"
unfiltered_variant_table = pd.read_csv(unfiltered_variant_region_path, sep="\t")
print(unfiltered_variant_table.shape[0]) # 85566
unfiltered_variant_table.head()

85566


,Region,ID
0,SKI|ENSG00000157933.11|EH38E2778468_fwd_tile1-1,REF_SKI|ENSG00000157933.11|EH38E2778468_fwd_ti...
1,SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1,REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_ti...
2,SKI|ENSG00000157933.11|EH38E2778473_fwd_tile1-1,REF_SKI|ENSG00000157933.11|EH38E2778473_fwd_ti...
3,SKI|ENSG00000157933.11|EH38E2778490_fwd_tile1-1,REF_SKI|ENSG00000157933.11|EH38E2778490_fwd_ti...
4,SKI|ENSG00000157933.11|EH38E2778492_fwd_tile1-1,REF_SKI|ENSG00000157933.11|EH38E2778492_fwd_ti...


In [23]:
# investigate final design region map
not_fitting_region = not_fitting_cases.merge(unfiltered_variant_table[['Region', 'ID']].drop_duplicates(), left_on="tmp_matching_header", right_on="ID", how="left")
merged_not_fitting_region = not_fitting_region[~not_fitting_region['Region'].isna()]
print("matched ref: ", merged_not_fitting_region.shape[0]) # 0

# alt_not_fitting_region = alt_not_fitting_cases.merge(unfiltered_variant_table[['Region', 'ALT_ID']].drop_duplicates(), left_on="tmp_matching_header", right_on="ALT_ID", how="left")
# merged_alt_not_fitting_region = alt_not_fitting_region[~alt_not_fitting_region['Region'].isna()]
# print("matched alt: ", merged_alt_not_fitting_region.shape[0]) # 0

matched ref:  0
